# Evaluate existing service performance

Read the operational records and service-intention mapping, check the selected jurisdictions, and estimate group and meta-frontier performance. The packaged case contains 200 DMUs. The DDF uses SciPy HiGHS and does not require a Gurobi licence. Complete this stage before fitting the Tobit models.

Select a Python kernel with the required packages; use the dependency profile in requirements.txt for the saved demand model. The reference installation uses the **ArcPy** kernel (Anaconda base). Run the cells in order. Shared settings are in `desktop_settings.json`; the default result folder is `../outputs/desktop_run_en`. Run the first setup cell after reopening an updated notebook so the helper code is refreshed. The scripts check dependencies without installing packages. Starting validation or a complete run preserves incompatible previous results in their original folder and selects a fresh run ID when required.

In [ ]:
from pathlib import Path
import sys
# Open notebooks within the toolkit; editor scripts can also locate their own folder.
_entry_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ROOT = next((p for p in [_entry_dir, *_entry_dir.parents, Path.cwd(), *Path.cwd().parents]
    if (p/'desktop_scripts/desktop_helpers.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the toolkit folder, then rerun this cell.')
sys.path.insert(0, str(ROOT/'desktop_scripts'))
import importlib
import desktop_helpers
importlib.reload(desktop_helpers)
from desktop_helpers import check_environment, run, run_r, get_output_dir, get_study_config
import pandas as pd
versions = check_environment()
OUT = get_output_dir()

Inspect the selected jurisdictions and service types.

In [ ]:
historical = pd.read_csv(ROOT/get_study_config()['paths']['historical'], dtype={'ServiceID': str})
print(historical.groupby('Jurisdiction').size())
print(historical.ServiceType.value_counts())

Run validation and historical evaluation. The mathematical implementation is in drt_toolkit/evaluation.py.

In [ ]:
OUT = run('validate', 'evaluate')
scores = pd.read_csv(OUT/'historical_scores.csv')
print(scores.head(12).to_string(index=False))

Fit the Tobit models by running this cell, or open 02_Performance_Drivers.R in RStudio and click Source. Use one method for this run ID.

In [ ]:
OUT = run_r('02_Performance_Drivers.R')
print((OUT/'tobit_summary.txt').read_text(encoding='utf-8',errors='replace'))